# Projeto 1 — NYC Taxi com Spark (baseline)

**Baseline:** implementação direta, sem tuning (gerada com IA, ver acknowledgements). De propósito:
- sem `cache`/`persist` do DataFrame limpo → cada operação relê e relimpa os Parquet
- sem `select` de colunas (column pruning manual)
- sem `broadcast()` explícito nos joins com a tabela de zonas
- `spark.sql.shuffle.partitions` e AQE com os valores por defeito

Operações: **M1, M2, M3** + **O1, O2**.

**Correr no Jupyter:** muda os valores por defeito na célula de parâmetros e faz *Run All*.

**Correr com spark-submit (cluster):** converte o notebook em script e passa os parâmetros:
```bash
jupyter nbconvert --to script main.ipynb        # gera main.py
spark-submit --master spark://spark-master:7077 main.py --years 2018 2019 2020 --fraction 0.1 --run-name baseline-10pct
```

## Parâmetros

In [1]:
import sys, argparse

p = argparse.ArgumentParser(description="NYC Taxi Spark pipeline")
p.add_argument("--master", default="spark://spark-master:7077",
               help="ex.: spark://spark-master:7077 ou local[*]")
p.add_argument("--input", default="/data",
               help="pasta com yellow_tripdata_YYYY-MM.parquet e taxi_zone_lookup.csv")
p.add_argument("--output", default="/data/output")
p.add_argument("--years", type=int, nargs="+", default=[2018, 2019, 2020])
p.add_argument("--months", type=int, nargs="+", default=list(range(1, 10)),
               help="ficheiros que não existam são ignorados")
p.add_argument("--fraction", type=float, default=1.0,
               help="fração dos registos (ex.: 0.1)")
p.add_argument("--ops", nargs="+", default=["M1", "M2", "M3", "O1", "O2"],
               choices=["M1", "M2", "M3", "O1", "O2"])
p.add_argument("--run-name", default="rdd_e_cache",
               help="etiqueta desta run no metrics.csv")
p.add_argument("--event-log-dir", default=None,
               help="se dado, ativa spark.eventLog (History Server)")

# No Jupyter usa os defaults acima; como script (spark-submit) lê a linha de comando
IN_NOTEBOOK = "ipykernel" in sys.modules
args = p.parse_args([] if IN_NOTEBOOK else None)
args

Namespace(master='spark://spark-master:7077', input='/data', output='/data/output', years=[2018, 2019, 2020], months=[1, 2, 3, 4, 5, 6, 7, 8, 9], fraction=1.0, ops=['M1', 'M2', 'M3', 'O1', 'O2'], run_name='rdd_e_cache', event_log_dir=None)

## Setup

In [2]:
import os, time, csv
from functools import reduce

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, IntegerType,
                               DoubleType, StringType, TimestampType)

builder = (SparkSession.builder
           .master(args.master)
           .appName(f"nyc-taxi-{args.run_name}")
           .config("spark.driver.memory", "2g")
           .config("spark.executor.memory", "1536m")   # abaixo do --memory 2G do worker
           .config("spark.ui.showConsoleProgress", "false"))
if args.event_log_dir:
    builder = (builder.config("spark.eventLog.enabled", "true")
                      .config("spark.eventLog.dir", args.event_log_dir))
spark = builder.getOrCreate()

print("Spark", spark.version, "| master:", spark.sparkContext.master,
      "| UI:", spark.sparkContext.uiWebUrl,
      "| cores:", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 20:53:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0 | master: spark://spark-master:7077 | UI: http://c15f0c0ac997:4040 | cores: 2


In [3]:
sys.path.append("/workspace/apps")
from spark_profiler import SparkProfiler
profiler = SparkProfiler(spark)

## M1 — Ingestão e limpeza
O schema muda entre ficheiros (ex.: `passenger_count` double vs long), por isso lê-se mês a mês e faz-se cast para um schema fixo.

In [4]:
TARGET_SCHEMA = StructType([
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("RatecodeID",            IntegerType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("payment_type",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("tip_amount",            DoubleType()),
    StructField("total_amount",          DoubleType()),
])

ZONES_SCHEMA = StructType([
    StructField("LocationID",   IntegerType()),
    StructField("Borough",      StringType()),
    StructField("Zone",         StringType()),
    StructField("service_zone", StringType()),
])

NULL_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
             "trip_distance", "fare_amount", "PULocationID", "DOLocationID"]


def to_target_schema(df, schema=TARGET_SCHEMA):
    """Cast de cada coluna para o tipo alvo; colunas em falta ficam null."""
    existing = set(df.columns)
    return df.select([
        (F.col(f.name) if f.name in existing else F.lit(None)).cast(f.dataType).alias(f.name)
        for f in schema.fields
    ])


files = [(os.path.join(args.input, f"yellow_tripdata_{y}-{m:02d}.parquet"), y)
         for y in args.years for m in args.months]
files = [(p_, y) for p_, y in files if os.path.exists(p_)]
assert files, "Nenhum ficheiro encontrado"
print(len(files), "ficheiros:", [os.path.basename(p_) for p_, _ in files])

zones = spark.read.csv(os.path.join(args.input, "taxi_zone_lookup.csv"),
                       header=True, schema=ZONES_SCHEMA)

from pyspark.sql import functions as F



colunas = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
           "trip_distance", "PULocationID", "DOLocationID", "fare_amount", "payment_type"]
tamanho = 0
for path, ano in files:
    tamanho += os.path.getsize(path)
MIB = 1024**2
alvo = 120 * MIB
workers = 6

import math

n_min = math.ceil(tamanho / alvo)
n_particoes = math.ceil(n_min / workers) * workers

raw_original = reduce(lambda a, b: a.unionByName(b),
             [to_target_schema(spark.read.parquet(p_))
                  .withColumn("file_year", F.lit(y))
                  .select(*colunas, "file_year")
              for p_, y in files])


import math
PREP = os.path.join(args.input, "trips_raw_balanced")

# 1) Preparação (uma vez): esquema já normalizado, ficheiros equilibrados
#    raw_original = o teu reduce(...) atual, SEM sample e SEM repartition
with profiler.profile("preparacao"):
    (raw_original
        .repartitionByRange(n_particoes, "tpep_pickup_datetime")
        .sortWithinPartitions("tpep_pickup_datetime")
        .write.mode("overwrite")
        .option("parquet.block.size", 32 * MIB)   # row groups de ~32 MiB
        .parquet(PREP))

# 2) Leitura (em cada run): uma única chamada, sem union nem problemas de esquema
tam_prep = sum(os.path.getsize(os.path.join(d, f))
               for d, _, fs in os.walk(PREP) for f in fs if f.endswith(".parquet"))
spark.conf.set("spark.sql.files.maxPartitionBytes", math.ceil(tam_prep / n_particoes))

raw = spark.read.parquet(PREP)
if args.fraction < 1.0:
    raw = raw.sample(withReplacement=False, fraction=args.fraction, seed=42)

print("partições de leitura:", raw.rdd.getNumPartitions())

27 ficheiros: ['yellow_tripdata_2018-01.parquet', 'yellow_tripdata_2018-02.parquet', 'yellow_tripdata_2018-03.parquet', 'yellow_tripdata_2018-04.parquet', 'yellow_tripdata_2018-05.parquet', 'yellow_tripdata_2018-06.parquet', 'yellow_tripdata_2018-07.parquet', 'yellow_tripdata_2018-08.parquet', 'yellow_tripdata_2018-09.parquet', 'yellow_tripdata_2019-01.parquet', 'yellow_tripdata_2019-02.parquet', 'yellow_tripdata_2019-03.parquet', 'yellow_tripdata_2019-04.parquet', 'yellow_tripdata_2019-05.parquet', 'yellow_tripdata_2019-06.parquet', 'yellow_tripdata_2019-07.parquet', 'yellow_tripdata_2019-08.parquet', 'yellow_tripdata_2019-09.parquet', 'yellow_tripdata_2020-01.parquet', 'yellow_tripdata_2020-02.parquet', 'yellow_tripdata_2020-03.parquet', 'yellow_tripdata_2020-04.parquet', 'yellow_tripdata_2020-05.parquet', 'yellow_tripdata_2020-06.parquet', 'yellow_tripdata_2020-07.parquet', 'yellow_tripdata_2020-08.parquet', 'yellow_tripdata_2020-09.parquet']
[preparacao] wall time: 78.78s (5 stages

In [5]:
def out_of_range(c):
    return (F.col(c) < 1) | (F.col(c) > 263)

# Critérios do enunciado + 2 extra (nulls; ano do pickup != ano do ficheiro,
# ex.: viagens com data de 2009 em ficheiros de 2018).
# coalesce(..., False): comparações com null dão null -> não contam nesse critério
# (os nulls já são apanhados por 'null_values').
INVALID = {
    "null_values":           reduce(lambda a, b: a | b, [F.col(c).isNull() for c in NULL_COLS]),
    "fare_amount_le_0":      F.col("fare_amount") <= 0,
    "trip_distance_le_0":    F.col("trip_distance") <= 0,
    "passenger_count_0_gt6": (F.col("passenger_count") == 0) | (F.col("passenger_count") > 6),
    "dropoff_le_pickup":     F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"),
    "location_out_of_range": out_of_range("PULocationID") | out_of_range("DOLocationID"),
    "year_mismatch":         F.year("tpep_pickup_datetime") != F.col("file_year"),
}
BAD = {n: F.coalesce(c, F.lit(False)) for n, c in INVALID.items()}
ANY_BAD = reduce(lambda a, b: a | b, BAD.values())

# DataFrame limpo — baseline:
trips = (raw.filter(~ANY_BAD)
         .drop("file_year")
         .cache())





In [6]:
with profiler.profile("M1"):
    # por ano: total, mantidos e nº que falha cada critério (podem sobrepor-se)
    aggs = [F.count("*").alias("total"), F.sum((~ANY_BAD).cast("int")).alias("kept")]
    aggs += [F.sum(c.cast("int")).alias(f"removed_{n}") for n, c in BAD.items()]
    m1 = raw.groupBy("file_year").agg(*aggs).orderBy("file_year")

    m1_pdf = m1.toPandas()
    m1_pdf.to_csv(os.path.join(args.output, "m1_cleaning_stats.csv"), index=False)

m1_pdf.set_index("file_year").T

/opt/spark/python/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


[M1] wall time: 12.11s (8 stages, 108 tasks) - see profiler.report() / profiler.compare(...)


file_year,2018,2019,2020
total,77685743,63610125,19997062
kept,75260335,60803736,18601319
removed_null_values,13830,299151,505549
removed_fare_amount_le_0,69071,137961,79431
removed_trip_distance_le_0,514169,537093,271606
removed_passenger_count_0_gt6,603509,1133801,399931
removed_dropoff_le_pickup,64730,63313,18293
removed_location_out_of_range,1438843,935376,234574
removed_year_mismatch,1003,990,231


## M2 — Procura temporal
Por (pickup_borough, day_of_week, hour_of_day): nº de viagens, tarifa média, duração média (min), distância média. Escrito em Parquet particionado por borough.

In [7]:
with profiler.profile("M2"):
    m2 = (trips
          .join(zones, trips.PULocationID == zones.LocationID)
          .withColumn("pickup_borough", F.col("Borough"))
          .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
          .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
          .withColumn("duration_min",
                      (F.unix_timestamp("tpep_dropoff_datetime")
                       - F.unix_timestamp("tpep_pickup_datetime")) / 60.0)
          .groupBy("pickup_borough", "day_of_week", "hour_of_day")
          .agg(F.count("*").alias("trip_count"),
               F.avg("fare_amount").alias("mean_fare"),
               F.avg("duration_min").alias("mean_duration_min"),
               F.avg("trip_distance").alias("mean_distance")))

    m2.write.mode("overwrite").partitionBy("pickup_borough") \
        .parquet(os.path.join(args.output, "m2_temporal_demand"))
m2.explain(mode="formatted")

[M2] wall time: 47.03s (5 stages, 80 tasks) - see profiler.report() / profiler.compare(...)
== Physical Plan ==
AdaptiveSparkPlan (16)
+- HashAggregate (15)
   +- Exchange (14)
      +- HashAggregate (13)
         +- Project (12)
            +- BroadcastHashJoin Inner BuildRight (11)
               :- Filter (7)
               :  +- InMemoryTableScan (1)
               :        +- InMemoryRelation (2)
               :              +- * Project (6)
               :                 +- * Filter (5)
               :                    +- * ColumnarToRow (4)
               :                       +- Scan parquet  (3)
               +- BroadcastExchange (10)
                  +- Filter (9)
                     +- Scan csv  (8)


(1) InMemoryTableScan
Output [5]: [tpep_pickup_datetime#868, tpep_dropoff_datetime#869, trip_distance#871, PULocationID#872, fare_amount#874]
Arguments: [tpep_pickup_datetime#868, tpep_dropoff_datetime#869, trip_distance#871, PULocationID#872, fare_amount#874], [isno

## M3 — Estatísticas por zona
Por zona e mês: média e desvio-padrão de fare e distância. Windows: rank no borough, quota rolante de 3 meses no borough, e flag quando o volume se afasta mais de 1.5σ da média rolante da própria zona (3 meses anteriores).

In [8]:
with profiler.profile("M3"):
    zone_month = (trips
        .join(zones, trips.PULocationID == zones.LocationID, "inner")
        .withColumn("year", F.year("tpep_pickup_datetime"))
        .withColumn("month", F.month("tpep_pickup_datetime"))
        .groupBy("Borough", "PULocationID", "Zone", "year", "month")
        .agg(F.count("*").alias("trip_count"),
             F.avg("fare_amount").alias("mean_fare"),
             F.stddev("fare_amount").alias("std_fare"),
             F.avg("trip_distance").alias("mean_distance"),
             F.stddev("trip_distance").alias("std_distance"))
        # índice contínuo de mês -> rangeBetween respeita meses em falta
        .withColumn("month_idx", F.col("year") * 12 + F.col("month")))

    # 1) rank por volume dentro do borough, em cada mês
    w_rank = Window.partitionBy("Borough", "year", "month").orderBy(F.desc("trip_count"))

    # 2) quota rolante 3 meses = viagens da zona (m-2..m) / viagens do borough (m-2..m)
    w_zone_3m = Window.partitionBy("PULocationID").orderBy("month_idx").rangeBetween(-2, 0)
    w_boro_3m = Window.partitionBy("Borough").orderBy("month_idx").rangeBetween(-2, 0)
    borough_roll = (zone_month.groupBy("Borough", "month_idx")
                    .agg(F.sum("trip_count").alias("borough_count"))
                    .withColumn("borough_3m", F.sum("borough_count").over(w_boro_3m))
                    .drop("borough_count"))

    # 3) desvio face à média rolante da própria zona (3 meses anteriores)
    w_prev = Window.partitionBy("PULocationID").orderBy("month_idx").rangeBetween(-3, -1)

    m3 = (zone_month
        .withColumn("rank_in_borough", F.rank().over(w_rank))
        .withColumn("zone_3m", F.sum("trip_count").over(w_zone_3m))
        .join(borough_roll, ["Borough", "month_idx"])
        .withColumn("share_3m", F.col("zone_3m") / F.col("borough_3m"))
        .withColumn("roll_mean", F.avg("trip_count").over(w_prev))
        .withColumn("roll_std", F.stddev("trip_count").over(w_prev))
        .withColumn("volume_anomaly",
                    F.coalesce(F.abs(F.col("trip_count") - F.col("roll_mean"))
                               > 1.5 * F.col("roll_std"), F.lit(False))))

    m3.write.mode("overwrite").parquet(os.path.join(args.output, "m3_zone_stats"))

    m3.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [Borough#1, month_idx#1445, PULocationID#872, Zone#2, year#1387, month#1388, trip_count#1389L, mean_fare#1390, std_fare#1391, mean_distance#1392, std_distance#1393, rank_in_borough#1461, zone_3m#1470L, borough_3m#1459L, share_3m#1527, roll_mean#1528, roll_std#1530, coalesce((abs((cast(trip_count#1389L as double) - roll_mean#1528)) > (roll_std#1530 * 1.5)), false) AS volume_anomaly#1557]
   +- Window [stddev(_w0#1548) windowspecdefinition(PULocationID#872, month_idx#1445 ASC NULLS FIRST, specifiedwindowframe(RangeFrame, -3, -1)) AS roll_std#1530], [PULocationID#872], [month_idx#1445 ASC NULLS FIRST]
      +- Project [Borough#1, month_idx#1445, PULocationID#872, Zone#2, year#1387, month#1388, trip_count#1389L, mean_fare#1390, std_fare#1391, mean_distance#1392, std_distance#1393, rank_in_borough#1461, zone_3m#1470L, borough_3m#1459L, share_3m#1527, roll_mean#1528, cast(trip_count#1389L as double) AS _w0#1548]
         +- W

## O1 — Anomalias de tarifa
Por (PULocationID, hour_of_day): média e σ de fare/km; flag quando fare/km > média + 3σ; taxa de anomalias dia útil vs fim de semana por payment_type e borough.

In [9]:
with profiler.profile("O1"):
    # trip_distance está em milhas -> converter para km
    o1_base = (trips
        .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
        .withColumn("is_weekend", F.dayofweek("tpep_pickup_datetime").isin(1, 7))
        .withColumn("fare_per_km", F.col("fare_amount") / (F.col("trip_distance") * 1.609344)))

    bucket_stats = (o1_base.groupBy("PULocationID", "hour_of_day")
                    .agg(F.avg("fare_per_km").alias("fpk_mean"),
                         F.stddev("fare_per_km").alias("fpk_std")))

    flagged = (o1_base
        .join(bucket_stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly",
                    F.coalesce(F.col("fare_per_km") > F.col("fpk_mean") + 3 * F.col("fpk_std"),
                               F.lit(False))))

    o1 = (flagged
        .join(zones, flagged.PULocationID == zones.LocationID, "left")
        .withColumn("pickup_borough", F.col("Borough"))
        .groupBy("pickup_borough", "payment_type", "is_weekend")
        .agg(F.count("*").alias("trips"),
             F.sum(F.col("is_anomaly").cast("int")).alias("anomalies"),
             F.avg(F.col("is_anomaly").cast("int")).alias("anomaly_rate")))

    o1.write.mode("overwrite").parquet(os.path.join(args.output, "o1_fare_anomalies"))

[O1] wall time: 30.63s (10 stages, 190 tasks) - see profiler.report() / profiler.compare(...)


## O2 — Fluxos origem-destino
Contagens mensais por par (PU, DO) com join duplo às zonas; quota de cada rota nas saídas da zona; top-20 rotas por borough; rotas cuja quota inverte a tendência entre Q1 (jan→mar) e Q3 (jul→set).

⚠️ A parte Q1 vs Q3 precisa de dados de julho e setembro.

In [10]:
with profiler.profile("O2"):
    pu = zones.select(F.col("LocationID").alias("PULocationID"),
                      F.col("Borough").alias("pickup_borough"),
                      F.col("Zone").alias("pickup_zone"))
    do = zones.select(F.col("LocationID").alias("DOLocationID"),
                      F.col("Borough").alias("dropoff_borough"),
                      F.col("Zone").alias("dropoff_zone"))

    od = (trips
        .withColumn("year", F.year("tpep_pickup_datetime"))
        .withColumn("month", F.month("tpep_pickup_datetime"))
        .groupBy("PULocationID", "DOLocationID", "year", "month")
        .agg(F.count("*").alias("trips"))
        .join(pu, "PULocationID", "left")
        .join(do, "DOLocationID", "left"))

    # quota da rota no total de saídas da zona de pickup, por mês
    w_out = Window.partitionBy("PULocationID", "year", "month")
    od = od.withColumn("share", F.col("trips") / F.sum("trips").over(w_out))
    od.write.mode("overwrite").parquet(os.path.join(args.output, "o2_od_monthly"))

    # top-20 rotas por borough de pickup (volume total no período)
    w_top = Window.partitionBy("pickup_borough").orderBy(F.desc("total_trips"))
    top20 = (od.groupBy("pickup_borough", "PULocationID", "pickup_zone",
                        "DOLocationID", "dropoff_zone")
               .agg(F.sum("trips").alias("total_trips"))
               .withColumn("rank", F.row_number().over(w_top))
               .filter(F.col("rank") <= 20))
    top20.write.mode("overwrite").parquet(os.path.join(args.output, "o2_top20_routes"))

    # inversão de tendência: sinal(mar - jan) vs sinal(set - jul), por rota e ano
    reversals = (od.groupBy("PULocationID", "DOLocationID", "year")
        .agg(F.max(F.when(F.col("month") == 1, F.col("share"))).alias("s_jan"),
             F.max(F.when(F.col("month") == 3, F.col("share"))).alias("s_mar"),
             F.max(F.when(F.col("month") == 7, F.col("share"))).alias("s_jul"),
             F.max(F.when(F.col("month") == 9, F.col("share"))).alias("s_sep"))
        .withColumn("q1_trend", F.signum(F.col("s_mar") - F.col("s_jan")))
        .withColumn("q3_trend", F.signum(F.col("s_sep") - F.col("s_jul")))
        .filter(F.col("q1_trend") * F.col("q3_trend") < 0))
    reversals.write.mode("overwrite").parquet(os.path.join(args.output, "o2_q1_q3_reversals"))

[O2] wall time: 26.79s (30 stages, 375 tasks) - see profiler.report() / profiler.compare(...)


## Métricas da run
Acrescenta uma linha por operação a `metrics.csv` (para comparar baseline vs otimizações e as runs de escalabilidade).

In [11]:
rep = profiler.report()
print(rep.to_string())

conf = {"run_name": args.run_name,
        "years": " ".join(map(str, args.years)),
        "fraction": args.fraction,
        "master": spark.sparkContext.master,
        "default_parallelism": spark.sparkContext.defaultParallelism,
        "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
        "aqe": spark.conf.get("spark.sql.adaptive.enabled")}

metrics_path = os.path.join(args.output, "metrics.csv")
rows = [{**conf, "step": name, **m} for name, m in profiler.runs.items()]
rows.append({**conf, "step": "TOTAL",
             "wall_time_s": round(sum(m["wall_time_s"] for m in profiler.runs.values()), 2)})
# pandas junta as colunas novas do profiler às runs antigas (que ficam vazias nessas colunas)
import pandas as pd
new_df = pd.DataFrame(rows)
if os.path.exists(metrics_path):
    new_df = pd.concat([pd.read_csv(metrics_path), new_df], ignore_index=True)
new_df.to_csv(metrics_path, index=False)
print("Métricas acrescentadas a", metrics_path)

            wall_time_s  num_jobs  num_stages  num_tasks  executor_run_time_ms  executor_cpu_time_ms input_bytes output_bytes shuffle_read_bytes shuffle_write_bytes memory_spilled_bytes disk_spilled_bytes  tasks_executed  input_records  files_read files_read_bytes  read_tasks  read_tasks_empty  read_task_median_ms  read_task_max_ms  shuffle_stages  shuffle_partitions_max  shuffle_skew_max  gc_time_ms
run                                                                                                                                                                                                                                                                                                                                                                                                                
preparacao        78.78         4           5        485                429930                363841       3.1MB        2.0GB              4.6GB               4.6GB               11.2GB       

### Distribuição do trabalho (para comparar com a versão otimizada)
- `read_tasks_empty`: tarefas de leitura que leram 0 linhas
- `read_task_median_ms` vs `read_task_max_ms`: desequilíbrio da leitura
- `shuffle_skew_max`: maior partição de shuffle / mediana (1 = equilibrado)
- `files_read_bytes`: tamanho dos ficheiros abertos (o `input_bytes` vem subcontado com parquet local)

In [12]:
cols = ["wall_time_s", "tasks_executed", "read_tasks", "read_tasks_empty",
        "read_task_median_ms", "read_task_max_ms", "files_read", "files_read_bytes",
        "input_records", "shuffle_write_bytes", "shuffle_skew_max",
        "memory_spilled_bytes", "disk_spilled_bytes", "gc_time_ms", "executor_run_time_ms"]
rep[[c for c in cols if c in rep.columns]]

,wall_time_s,tasks_executed,read_tasks,read_tasks_empty,read_task_median_ms,read_task_max_ms,files_read,files_read_bytes,input_records,shuffle_write_bytes,shuffle_skew_max,memory_spilled_bytes,disk_spilled_bytes,gc_time_ms,executor_run_time_ms
run,,,,,,,,,,,,,,,
preparacao,78.78,338,314,240,47,5800,27,2.2GB,476586103,4.6GB,1.26,11.2GB,3.7GB,14494,429930
M1,12.11,29,26,1,2086,5466,24,2.0GB,161292930,4.3KB,1.00,0.0B,0.0B,3475,61319
M2,47.03,54,52,2,3672,12170,25,2.0GB,161308675,1.4MB,1.00,0.0B,0.0B,10960,270884
M3,25.11,61,52,2,2768,4451,2,24.0KB,31490,4.0MB,1.00,0.0B,0.0B,5452,138046
O1,30.63,86,52,2,1979,5475,1,12.0KB,31225,1.6GB,1.22,0.0B,0.0B,5220,175721
O2,26.79,125,78,3,1552,2520,4,48.0KB,47500,101.9MB,1.10,0.0B,0.0B,5908,138957


In [13]:
import sys
print(sys.executable)

/usr/bin/python3


In [14]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 27.8 MB/s eta 0:00:00


0

In [15]:
import pyarrow.parquet as pq
for p_, _ in files[:5]:
    md = pq.ParquetFile(p_).metadata
    print(os.path.basename(p_), "row groups:", md.num_row_groups, "linhas:", md.num_rows)

yellow_tripdata_2018-01.parquet row groups: 1 linhas: 8760687
yellow_tripdata_2018-02.parquet row groups: 1 linhas: 8492819
yellow_tripdata_2018-03.parquet row groups: 1 linhas: 9431289
yellow_tripdata_2018-04.parquet row groups: 1 linhas: 9306216
yellow_tripdata_2018-05.parquet row groups: 1 linhas: 9224788


## Planos de execução (para o relatório)
Procurar: `SortMergeJoin` vs `BroadcastHashJoin`, `Exchange`, colunas lidas no `FileScan`, nº de scans aos Parquet.

In [16]:
if IN_NOTEBOOK:
    m2.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (16)
+- HashAggregate (15)
   +- Exchange (14)
      +- HashAggregate (13)
         +- Project (12)
            +- BroadcastHashJoin Inner BuildRight (11)
               :- Filter (7)
               :  +- InMemoryTableScan (1)
               :        +- InMemoryRelation (2)
               :              +- * Project (6)
               :                 +- * Filter (5)
               :                    +- * ColumnarToRow (4)
               :                       +- Scan parquet  (3)
               +- BroadcastExchange (10)
                  +- Filter (9)
                     +- Scan csv  (8)


(1) InMemoryTableScan
Output [5]: [tpep_pickup_datetime#868, tpep_dropoff_datetime#869, trip_distance#871, PULocationID#872, fare_amount#874]
Arguments: [tpep_pickup_datetime#868, tpep_dropoff_datetime#869, trip_distance#871, PULocationID#872, fare_amount#874], [isnotnull(PULocationID#872)]

(2) InMemoryRelation
Arguments: [tpep_pickup_datetime#868, tpep_dr

In [17]:
if IN_NOTEBOOK:
    o1.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (29)
+- HashAggregate (28)
   +- Exchange (27)
      +- HashAggregate (26)
         +- Project (25)
            +- BroadcastHashJoin LeftOuter BuildRight (24)
               :- Project (20)
               :  +- SortMergeJoin Inner (19)
               :     :- Sort (10)
               :     :  +- Exchange (9)
               :     :     +- Project (8)
               :     :        +- Filter (7)
               :     :           +- InMemoryTableScan (1)
               :     :                 +- InMemoryRelation (2)
               :     :                       +- * Project (6)
               :     :                          +- * Filter (5)
               :     :                             +- * ColumnarToRow (4)
               :     :                                +- Scan parquet  (3)
               :     +- Sort (18)
               :        +- HashAggregate (17)
               :           +- Exchange (16)
               :              +- HashAggregate

In [18]:
if IN_NOTEBOOK:
    od.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (27)
+- Project (26)
   +- Window (25)
      +- Sort (24)
         +- Exchange (23)
            +- Project (22)
               +- BroadcastHashJoin LeftOuter BuildRight (21)
                  :- Project (16)
                  :  +- BroadcastHashJoin LeftOuter BuildRight (15)
                  :     :- HashAggregate (10)
                  :     :  +- Exchange (9)
                  :     :     +- HashAggregate (8)
                  :     :        +- Project (7)
                  :     :           +- InMemoryTableScan (1)
                  :     :                 +- InMemoryRelation (2)
                  :     :                       +- * Project (6)
                  :     :                          +- * Filter (5)
                  :     :                             +- * ColumnarToRow (4)
                  :     :                                +- Scan parquet  (3)
                  :     +- BroadcastExchange (14)
                  :        +- Proj

## Ver os resultados

In [19]:
if IN_NOTEBOOK:
    out = args.output
    spark.read.parquet(f"{out}/m2_temporal_demand").orderBy("pickup_borough", "day_of_week", "hour_of_day").show(10)
    spark.read.parquet(f"{out}/m3_zone_stats").filter("volume_anomaly").show(10)
    spark.read.parquet(f"{out}/o1_fare_anomalies").orderBy("pickup_borough", "payment_type", "is_weekend").show(10)
    spark.read.parquet(f"{out}/o2_top20_routes").orderBy("pickup_borough", "rank").show(10)

+-----------+-----------+----------+------------------+------------------+------------------+--------------+
|day_of_week|hour_of_day|trip_count|         mean_fare| mean_duration_min|     mean_distance|pickup_borough|
+-----------+-----------+----------+------------------+------------------+------------------+--------------+
|          1|          0|      1190| 16.45740336134454|15.877857142857142| 4.342252100840336|         Bronx|
|          1|          1|      1039|15.297314725697786|21.191498235482843| 4.019201154956689|         Bronx|
|          1|          2|       891|14.375959595959596| 14.50581743359521|3.9568462401795728|         Bronx|
|          1|          3|       943|13.664612937433722|15.755478967833158| 3.649745493107106|         Bronx|
|          1|          4|       992|14.634153225806452|17.890238575268814| 4.036572580645162|         Bronx|
|          1|          5|       746| 15.72642091152815|15.253574620196604| 4.552949061662199|         Bronx|
|          1|      

In [20]:
if not IN_NOTEBOOK:
    spark.stop()